# AE + Contrastive (SimCLR) + MNA ile Açı+Hız Eğitimi

**Amaç:** Önceden çıkarılmış XY `.npy` dosyalarını kullanarak:
- Üst beden açılar (6 kanal) + **açısal hızlar** (6 kanal) ⇒ toplam **12 özellik** çıkar,
- **Pencereleme** (W=30, stride=15) ve **z-score** normalizasyon uygula,
- **Encoder: 1D-CNN → GAP → FC**, **Decoder: Multi-head (MNA)**, + **SimCLR (NT-Xent)** ile eğit,
- (Opsiyonel) başarı yüzdesi ve latent çıkarım.

 Neden **1D-CNN**? Zaman eksenindeki lokal desenleri hızlı ve stabil yakalar.  
 Neden **GAP**? Zaman boyutunu özetleyip parametreyi azaltır, uzunluğa karşı dayanıklıdır.


## 1.Kurulum, yollar, hiperparametreler

In [ ]:
import os, re, glob, random
from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Yol/klasör ayarları (otomatik .npy bulma/düzeltme içerir) 
BASE = r"C:\Users\The Coder Farmer\Desktop\Autoencoder_video"  # <-- KENDİ YOLUN

# Varsayılan yollar
DIR_TR = os.path.join(BASE, "npy_cikti")        # eğitim .npy (XY)
DIR_TE = os.path.join(BASE, "npy_cikti_test")   # test .npy (opsiyonel)
OUTDIR = os.path.join(BASE, "ae_mna_pose_angles_out")
os.makedirs(OUTDIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# --- .npy arayıcı (hem .npy hem .NPY yakalar) ---
def _list_npy_case_insensitive(folder: str):
    return (glob.glob(os.path.join(folder, "*.npy")) +
            glob.glob(os.path.join(folder, "*.[nN][pP][yY]")))

print("BASE:", BASE)
print("Başlangıç DIR_TR:", DIR_TR)

# Eğer DIR_TR içinde .npy yoksa, BASE altında en çok .npy olan klasörü otomatik seç
cand_dir = DIR_TR
if (not os.path.isdir(cand_dir)) or (len(_list_npy_case_insensitive(cand_dir)) == 0):
    counts = {}
    for p in Path(BASE).rglob("*.[nN][pP][yY]"):
        d = str(p.parent)
        counts[d] = counts.get(d, 0) + 1
    assert counts, "BASE altında hiç .npy bulunamadı; BASE yolunu doğru klasöre ayarla."

    # 'test' içermeyen klasörleri tercih et (eğitim için)
    chosen = sorted(counts.items(), key=lambda x: -x[1])
    picked = None
    for d, c in chosen:
        if "test" not in d.lower():
            picked = d
            break
    if picked is None:
        picked = chosen[0][0]

    cand_dir = picked
    print("Uyarı: Başlangıç DIR_TR içinde .npy yoktu, otomatik seçildi ->", cand_dir)

# Son karar
DIR_TR = cand_dir
print("Kullanılacak DIR_TR:", DIR_TR, "| .npy sayısı:", len(_list_npy_case_insensitive(DIR_TR)))
print("DIR_TE:", DIR_TE, "| mevcut mu:", os.path.isdir(DIR_TE))

# ==== Hiperparametreler ====
WINDOW = 30
STRIDE = 15
BATCH_SIZE = 128
EPOCHS = 70
LR = 1e-3
LATENT_DIM = 128

# Contrastive (SimCLR)
LAMBDA_CONTR = 0.5
TEMP = 0.5

# MNA (komşu pencere rekonstrüksiyon)
OFFS = [-1, 1]
BETA_MNA = 0.3

VAL_SPLIT = 0.2
PATIENCE = 10


BASE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video
Başlangıç DIR_TR: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\npy_cikti
Kullanılacak DIR_TR: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\npy_cikti | .npy sayısı: 120
DIR_TE: C:\Users\The Coder Farmer\Desktop\Autoencoder_video\npy_cikti_test | mevcut mu: True


## 2. Yardımcı fonksiyonlar: dosya listeleme, pencereleme


In [102]:
def list_npy(folder: str) -> List[str]:
    files = glob.glob(os.path.join(folder, "*.npy"))
    def key_fn(p):
        m = re.findall(r"(\d+)", os.path.basename(p))
        return int(m[0]) if m else 10**9
    return sorted(files, key=key_fn)

def extract_id(p: str) -> int:
    m = re.findall(r"(\d+)", os.path.basename(p))
    return int(m[0]) if m else -1

def windowize(seq: np.ndarray, W: int, S: int) -> np.ndarray:
    T, F = seq.shape
    out = []
    for s in range(0, max(T - W + 1, 0), S):
        out.append(seq[s:s+W])
    return np.asarray(out, dtype=np.float32)


## 3. Üst beden 17 nokta → açı ve açısal hız üretimi
- Dirsek: ∠(omuz–dirsek–bilek)
- Omuz: ∠(kalça–omuz–dirsek)
- Gövde yaw: kalça hattı → omuz hattı signed açı
- Gövde eğimi: (kalça orta → omuz orta) vektörü ile dikey arasındaki açı
Toplam **6 açı**, + frame farkı ile **6 hız** ⇒ **12 kanal**.


In [103]:
# --- COCO-17 ve MediaPipe-33 için eklem eşlemeleri (üst beden) ---
COCO17 = dict(LShoulder=5, RShoulder=6, LElbow=7, RElbow=8, LWrist=9,  RWrist=10, LHip=11, RHip=12)
MP33   = dict(LShoulder=11, RShoulder=12, LElbow=13, RElbow=14, LWrist=15, RWrist=16, LHip=23, RHip=24)

def pick_joints(Fxy: int):
    """XY sütun sayısına (Fxy) göre 17/33 eklem eşlemesini seç."""
    pts = Fxy // 2
    if pts == 17: return COCO17
    if pts == 33: return MP33
    return None  # XY değilse None

# ---- yardımcılar ----
def angle_wrap(a):  # [-pi, pi]
    return (a + np.pi) % (2*np.pi) - np.pi

def angle_diff(a):  # ardışık fark (wrap duyarlı)
    da = np.diff(a, axis=0, prepend=a[[0], :])
    return angle_wrap(da)

def xy_to_points(frame_xy: np.ndarray):  # [Fxy] -> [N,2]
    return frame_xy.reshape(-1, 2)

def vec(p1, p2):  # p2 - p1
    return p2 - p1

def angle_between(u, v):  # 2D signed
    ang1 = np.arctan2(u[...,1], u[...,0])
    ang2 = np.arctan2(v[...,1], v[...,0])
    return angle_wrap(ang2 - ang1)

def angle_at(A, B, C):   # ∠ABC (0..pi)
    BA = A - B; BC = C - B
    num = (BA * BC).sum(axis=-1)
    den = (np.linalg.norm(BA,axis=-1) * np.linalg.norm(BC,axis=-1) + 1e-8)
    cosv = np.clip(num/den, -1.0, 1.0)
    return np.arccos(cosv)

# ---- XY -> açılar (6 kanal) ----
def compute_angles_xy(seq_xy: np.ndarray) -> np.ndarray:
    """XY [T, Fxy] -> açılar [T,6]
       sıra: [LE, RE, LS, RS, yaw, lean]
    """
    T, Fxy = seq_xy.shape
    J = pick_joints(Fxy)
    assert (Fxy % 2 == 0) and (J is not None), "XY formatı bekleniyordu (17 veya 33 nokta)."

    ang = np.zeros((T, 6), dtype=np.float32)
    for t in range(T):
        P  = xy_to_points(seq_xy[t])         # [N,2]
        LS = P[J["LShoulder"]]; RS = P[J["RShoulder"]]
        LE = P[J["LElbow"]];    RE = P[J["RElbow"]]
        LW = P[J["LWrist"]];    RW = P[J["RWrist"]]
        LH = P[J["LHip"]];      RH = P[J["RHip"]]

        # Dirsek açıları: ∠(Shoulder-Elbow-Wrist)
        a_le = angle_at(LS, LE, LW)
        a_re = angle_at(RS, RE, RW)
        # Omuz açıları: ∠(Hip-Shoulder-Elbow)
        a_ls = angle_at(LH, LS, LE)
        a_rs = angle_at(RH, RS, RE)
        # Gövde yaw: hip hattı → shoulder hattına signed açı
        v_hp = vec(LH, RH); v_sh = vec(LS, RS)
        yaw  = angle_between(v_hp, v_sh)
        # Gövde eğimi: (mid-hip → mid-shoulder) ile dikey arasındaki açı
        mid_hip = (LH + RH)/2.0; mid_sh = (LS + RS)/2.0
        v_tr = vec(mid_hip, mid_sh); v_axis = np.array([0.0, -1.0], np.float32)
        lean = angle_between(v_axis, v_tr)

        ang[t] = [a_le, a_re, a_ls, a_rs, yaw, lean]
    return ang

# ---- XY -> [sin(a), cos(a), d(a)] = 18 kanal  ----
# a: [LE, RE, LS, RS, yaw, lean] (radyan)
# d(a): wrap-aware ardışık fark (angle_diff zaten wrap'lı)

def build_features_from_xy(seq_xy: np.ndarray) -> np.ndarray:
    """
    Girdi:
      seq_xy : (T, Fxy) XY koordinatları (COCO-17: Fxy=34 | MP-33: Fxy=66)
    Çıktı:
      feats  : (T, 18) = [sin(a 6), cos(a 6), d(a) 6]
    Notlar:
      - sin/cos dönüşümü açı sarmalanma (±π) problemini ortadan kaldırır.
      - d(a) wrap-aware olduğu için π↔−π komşuluğunda sıçrama yapmaz.
    """
    # 6 açıyı çıkart (radyan)
    ang  = compute_angles_xy(seq_xy).astype(np.float32)   # (T, 6)
    # açısal hız (wrap-aware)
    dang = angle_diff(ang).astype(np.float32)             # (T, 6)
    # sin/cos dönüşümü
    s = np.sin(ang).astype(np.float32)                    # (T, 6)
    c = np.cos(ang).astype(np.float32)                    # (T, 6)
    # birleştir -> 18 kanal
    feats = np.concatenate([s, c, dang], axis=1).astype(np.float32)  # (T, 18)
    return feats


In [104]:
def detect_format(arr: np.ndarray) -> str:
    """(T,F) dizisi: XY (17/33 nokta) ise 'xy', değilse 'feat'."""
    if arr.ndim == 2 and arr.shape[1] % 2 == 0 and pick_joints(arr.shape[1]) is not None:
        return "xy"
    return "feat"

def build_features_from_numeric(seq: np.ndarray) -> np.ndarray:
    """Önceden çıkarılmış özellik/ açı serileri için:
       feats = [ham, 1. fark (hız), 2. fark (ivme)]  -> (T, 3*F)
    """
    d1 = np.diff(seq, axis=0, prepend=seq[[0], :])   # velocity
    d2 = np.diff(d1, axis=0, prepend=d1[[0], :])     # acceleration
    return np.concatenate([seq, d1, d2], axis=1).astype(np.float32)


## 4. Dataset: pencereleme + z-score + SimCLR augment + MNA komşuları


In [105]:
# =======================
# Dataset sınıfı (GÜNCEL)
# =======================
class PoseWindowDataset(Dataset):
    """
    .npy dosyalarından pencere üretir.
      - XY (17/33 nokta) ise: 6 açı için [sin, cos] + açısal hız -> C = 18
      - Özellik/angle serisi ise: [ham, 1.fark, 2.fark] -> C = 3*F
    Dönen tensör şekilleri:
      x_clean, x1, x2: (C, L)   | neighs: (K, C, L) | mask: (K,)
    """
    def __init__(self, files: List[str], window=30, stride=15,
                 fit_stats=True, stats=None):
        self.files = files
        self.window = window
        self.stride = stride

        self.all_wins = []   # [N, W, C]
        self.meta = []       # global index -> (fid, pos)
        self.base = {}       # fid -> global start
        self.lenf = {}       # fid -> #wins

        gptr = 0
        for fp in self.files:
            arr = np.load(fp).astype(np.float32)     # (T, F)

            # --- FORMAT ALGILAMA ---
            fmt = detect_format(arr)                 # 'xy' veya 'feat'
            if fmt == "xy":
                feats = build_features_from_xy(arr)       # (T, 18)
            else:
                feats = build_features_from_numeric(arr)  # (T, 3*F)

            # --- PENCERELEME ---
            wins = windowize(feats, self.window, self.stride)  # (nW, W, C)
            fid = extract_id(fp)
            self.base[fid] = gptr
            self.lenf[fid] = len(wins)
            for pos in range(len(wins)):
                self.all_wins.append(wins[pos])
                self.meta.append((fid, pos))
                gptr += 1

        self.all_wins = np.asarray(self.all_wins, dtype=np.float32)  # (N, W, C)
        if self.all_wins.size == 0:
            raise RuntimeError("Pencere üretilemedi. WINDOW/STRIDE veya veri uzunluğunu kontrol et.")

        # --- Z-SCORE NORMALİZASYON ---
        if stats is None and fit_stats:
            self.mean = self.all_wins.mean(axis=(0, 1), keepdims=True)
            self.std  = self.all_wins.std(axis=(0, 1), keepdims=True) + 1e-8
        else:
            self.mean, self.std = stats
        self.all_wins = (self.all_wins - self.mean) / self.std

    def get_stats(self):
        return (self.mean, self.std)

    # ------ SimCLR augment'leri ------
    def _aug_jitter(self, x, sigma=0.02):
        return x + np.random.normal(0, sigma, size=x.shape).astype(np.float32)

    def _aug_scale(self, x, smin=0.85, smax=1.15):  # aralığı genişlettik
        s = np.random.uniform(smin, smax)
        return (x * s).astype(np.float32)

    def _aug_time_mask(self, x, max_len=6):         # maske süresi biraz daha uzun
        x = x.copy(); L = x.shape[0]
        m = np.random.randint(1, max_len+1)
        s = np.random.randint(0, max(1, L - m + 1))
        x[s:s+m] = 0.0
        return x

    def _aug_shift(self, x, max_shift=4):
        sh = np.random.randint(-max_shift, max_shift+1)
        return np.roll(x, sh, axis=0).astype(np.float32)

    def _aug_time_warp(self, x, warp_min=0.9, warp_max=1.1):
        """
        Zaman ölçeği değişimi (hızlandır/yavaşlat), L sabit kalacak.
        Basit lineer enterpolasyonla iki aşama: warp -> tekrar L'e ölçekle.
        """
        L, C = x.shape
        rate = np.random.uniform(warp_min, warp_max)
        # önce yeni uzunluğa örnekle
        new_L = max(2, int(round(L * rate)))
        t_src = np.linspace(0, 1, L)
        t_new = np.linspace(0, 1, new_L)
        x_w = np.vstack([np.interp(t_new, t_src, x[:, k]) for k in range(C)]).T  # (new_L, C)
        # tekrar L'e geri ölçekle
        t_back = np.linspace(0, 1, L)
        x_out = np.vstack([np.interp(t_back, np.linspace(0,1,new_L), x_w[:, k]) for k in range(C)]).T
        return x_out.astype(np.float32)

    def _apply_augs(self, x):
        funcs = [self._aug_jitter, self._aug_scale, self._aug_time_mask, self._aug_shift, self._aug_time_warp]
        fs = np.random.choice(funcs, size=2, replace=False)
        y = x
        for f in fs: y = f(y)
        return y.astype(np.float32)

    def __len__(self):
        return len(self.all_wins)

    def __getitem__(self, idx):
        x = self.all_wins[idx]          # (W, C)
        fid, pos = self.meta[idx]

        # --- MNA komşuları ---
        neighs, mask = [], []
        for d in OFFS:
            q = pos + d
            if 0 <= q < self.lenf[fid]:
                g = self.base[fid] + q
                neighs.append(self.all_wins[g])     # (W, C)
                mask.append(1.0)
            else:
                neighs.append(np.zeros_like(x))
                mask.append(0.0)
        neighs = np.stack(neighs, axis=0)           # (K, W, C)
        mask   = np.asarray(mask, np.float32)       # (K,)

        # --- SimCLR: iki görünüm ---
        x1 = self._apply_augs(x)
        x2 = self._apply_augs(x)

        # PyTorch düzeni: (C, L)
        x_clean = torch.from_numpy(x.T.copy())                        # (C, W)
        x1      = torch.from_numpy(x1.T.copy())
        x2      = torch.from_numpy(x2.T.copy())
        neighs  = torch.from_numpy(neighs.transpose(0, 2, 1).copy())  # (K, C, W)
        mask    = torch.from_numpy(mask)

        return x_clean, x1, x2, neighs, mask


In [106]:
ds_tr = PoseWindowDataset(train_files, WINDOW, STRIDE, fit_stats=True)
print("Train windows:", ds_tr.all_wins.shape)   # (N, W, C)
print("CIN:", ds_tr.all_wins.shape[-1])


Train windows: (584, 30, 6)
CIN: 6


## 5. Model: Encoder (1D-CNN → GAP → FC) + Decoder (center + komşular)
- 1D-CNN zaman eksenindeki lokal paternleri yakalar,
- GAP zaman boyunca özet çıkarır,
- FC latent vektörü üretir,
- Decoder, merkez pencere ve her komşu için ayrı head kullanır (MNA).


In [107]:
class MetricAE_MNA(nn.Module):
    """
    AE + MNA + SimCLR (projection head'li)
      - Encoder: 1D-CNN -> GAP -> FC (latent z)
      - Projection head: MLP (g(z))  -> contrastive bununla hesaplanır
      - Decoder'lar: merkez + her komşu ofset için ayrı linear
    """
    def __init__(self, in_channels: int, seq_len: int, latent_dim: int, offsets: list[int]):
        super().__init__()
        self.in_channels = in_channels
        self.seq_len = seq_len
        # ofset anahtarlarını string saklıyoruz (ModuleDict gereği)
        self.offsets = [str(d) for d in offsets]

        # --- Encoder ---
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 128,      kernel_size=5, padding=2),     nn.ReLU(),
            nn.Conv1d(128, 128,     kernel_size=3, padding=1),     nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )
        self.fc_mu = nn.Linear(128, latent_dim)  # latent z

        # --- Projection head (SimCLR) ---
        self.proj_head = nn.Sequential(
            nn.Linear(latent_dim, latent_dim), nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )

        # --- Decoder'lar ---
        self.dec_center = nn.Linear(latent_dim, in_channels * seq_len)
        self.dec_neighs = nn.ModuleDict({
            k: nn.Linear(latent_dim, in_channels * seq_len) for k in self.offsets
        })

    # ---------- yardımcılar ----------
    def _reshape(self, y):
        # (B, C*L) -> (B, C, L)
        return y.view(-1, self.in_channels, self.seq_len)

    # ---------- ileri yayılım ----------
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B,C,L) -> z: (B,D)"""
        h = self.encoder(x)
        z = self.fc_mu(h)
        return z

    def project(self, z: torch.Tensor) -> torch.Tensor:
        """z: (B,D) -> g(z): (B,D) (contrastive bununla)"""
        return self.proj_head(z)

    def decode_center(self, z: torch.Tensor) -> torch.Tensor:
        """z -> merkez pencere rekonstrüksiyonu (B,C,L)"""
        return self._reshape(self.dec_center(z))

    def decode_neighs(self, z: torch.Tensor) -> dict:
        """z -> komşu pencerelerin rekonstrüksiyonları dict[str]->(B,C,L)"""
        return {k: self._reshape(self.dec_neighs[k](z)) for k in self.offsets}

    def forward(self, x: torch.Tensor):
        """
        Dönüş:
          y0: (B,C,L)  -> merkez rekonstrüksiyon
          yN: dict[str]->(B,C,L) -> her komşu için rekonstrüksiyon
          z : (B,D)    -> latent
          p : (B,D)    -> projection head çıktısı (contrastive için)
        """
        z = self.encode(x)
        p = self.project(z)
        y0 = self.decode_center(z)
        yN = self.decode_neighs(z)
        return y0, yN, z, p


## 6. Contrastive loss (SimCLR / NT-Xent)


In [108]:
def nt_xent(z1, z2, temp=0.2):
    z1 = nn.functional.normalize(z1, dim=1)
    z2 = nn.functional.normalize(z2, dim=1)
    B = z1.size(0)
    reps = torch.cat([z1, z2], dim=0)          # (2B, D)
    sim  = torch.matmul(reps, reps.T) / temp   # (2B, 2B)
    mask = torch.eye(2*B, dtype=torch.bool, device=reps.device)
    sim  = sim.masked_fill(mask, -1e9)
    targets = torch.cat([torch.arange(B,2*B), torch.arange(0,B)]).to(reps.device)
    return nn.CrossEntropyLoss()(sim, targets)


## 7. Data split + DataLoader


In [109]:
# .npy dosyalarının şekillerini örnekle (XY mi kontrol)
files = list_npy(DIR_TR)
print("Toplam dosya:", len(files))
for fp in files[:5]:
    arr = np.load(fp, mmap_mode="r")
    print(os.path.basename(fp), "->", arr.shape)

# Beklenti:
#  - 17 nokta XY ise: (T, 34)
#  - MediaPipe 33 nokta XY ise: (T, 66)
#  Not: Son boyut çift olmalı (XY)


Toplam dosya: 60
angles_1.npy -> (259, 2)
angles_2.npy -> (273, 2)
angles_3.npy -> (173, 2)
angles_4.npy -> (231, 2)
angles_5.npy -> (197, 2)


In [110]:
all_files = list_npy(DIR_TR)
assert len(all_files) > 0, "Eğitim klasöründe .npy yok."

n_val = max(1, int(len(all_files)*VAL_SPLIT))
val_files = all_files[-n_val:]; train_files = all_files[:-n_val]

ds_tr = PoseWindowDataset(train_files, WINDOW, STRIDE, fit_stats=True)
stats = ds_tr.get_stats()
ds_va = PoseWindowDataset(val_files, WINDOW, STRIDE, fit_stats=False, stats=stats)

CIN = ds_tr.all_wins.shape[-1]   # 12
LSEQ = WINDOW

dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

model = MetricAE_MNA(CIN, LSEQ, LATENT_DIM, OFFS).to(DEVICE)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
mse   = nn.MSELoss()


## 8. Eğitim döngüsü (MSE_center + β*MSE_neighbors + λ*NT-Xent)
- Erken durdurma (PATIENCE) ile en iyi modeli kaydeder.


In [111]:
from torch.serialization import add_safe_globals
add_safe_globals([np.core.multiarray._reconstruct])  # PyTorch 2.6 güvenliği

best_val = float("inf"); pat = 0
BEST = os.path.join(OUTDIR, "best_mna.pt")

for epoch in range(1, EPOCHS+1):
    model.train()
    tr_c = tr_n = tr_con = tr_tot = 0.0; n = 0

    for x_clean, x1, x2, neigh, mask in dl_tr:
        x_clean = x_clean.to(DEVICE)
        x1 = x1.to(DEVICE); x2 = x2.to(DEVICE)
        neigh = neigh.to(DEVICE); mask = mask.to(DEVICE)

        opt.zero_grad()

        # ---- MERKEZ + KOMŞULAR (forward 4 değer döndürür) ----
        y0, yN, zc, pc = model(x_clean)

        # center MSE
        loss_c = mse(y0, x_clean)

        # neighbors (maskeli MSE ort.)
        loss_n, msum = 0.0, 0.0
        for j, d in enumerate(OFFS):
            pred = yN[str(d)]
            tgt  = neigh[:, j]
            m    = mask[:, j]
            l = ((pred - tgt)**2).mean(dim=(1,2))  # (B,)
            loss_n += (l * m).sum()
            msum   += m.sum()
        loss_n = loss_n / (msum + 1e-8)

        # ---- CONTRASTIVE: projection head ile (g(z)) ----
        z1 = model.encode(x1); p1 = model.project(z1)
        z2 = model.encode(x2); p2 = model.project(z2)
        loss_con = nt_xent(p1, p2, temp=TEMP)  # eskiden z1,z2 idi → p1,p2 oldu

        loss = loss_c + BETA_MNA*loss_n + LAMBDA_CONTR*loss_con
        loss.backward()
        opt.step()

        bs = x_clean.size(0)
        tr_c   += loss_c.item()*bs
        tr_n   += loss_n.item()*bs
        tr_con += loss_con.item()*bs
        tr_tot += loss.item()*bs
        n += bs

    tr_c/=n; tr_n/=n; tr_con/=n; tr_tot/=n

    # ---------------- VALIDASYON ----------------
    model.eval()
    va_c = va_n = va_con = 0.0; nv = 0
    with torch.no_grad():
        for x_clean, x1, x2, neigh, mask in dl_va:
            x_clean = x_clean.to(DEVICE)
            x1 = x1.to(DEVICE); x2 = x2.to(DEVICE)
            neigh = neigh.to(DEVICE); mask = mask.to(DEVICE)

            y0, yN, zc, pc = model(x_clean)
            lc = mse(y0, x_clean)

            ln, msum = 0.0, 0.0
            for j, d in enumerate(OFFS):
                pred = yN[str(d)]
                tgt  = neigh[:, j]
                m    = mask[:, j]
                l = ((pred - tgt)**2).mean(dim=(1,2))
                ln += (l * m).sum(); msum += m.sum()
            ln = ln / (msum + 1e-8)

            z1 = model.encode(x1); p1 = model.project(z1)
            z2 = model.encode(x2); p2 = model.project(z2)
            lcon = nt_xent(p1, p2, temp=TEMP)

            bs = x_clean.size(0)
            va_c   += lc.item()*bs
            va_n   += ln.item()*bs
            va_con += lcon.item()*bs
            nv += bs

    va_c/=nv; va_n/=nv; va_con/=nv
    va_tot = va_c + BETA_MNA*va_n + LAMBDA_CONTR*va_con

    print(f"[{epoch:03d}] TR tot {tr_tot:.4f} | c {tr_c:.4f} | n {tr_n:.4f} | con {tr_con:.4f} "
          f"|| VA tot {va_tot:.4f} | c {va_c:.4f} | n {va_n:.4f} | con {va_con:.4f}")

    if va_tot < best_val - 1e-5:
        best_val = va_tot; pat = 0
        torch.save({
            "state_dict": model.state_dict(),
            "stats_mean": torch.from_numpy(ds_tr.get_stats()[0]).float(),
            "stats_std":  torch.from_numpy(ds_tr.get_stats()[1]).float(),
            "cin": CIN, "lseq": LSEQ, "latent": LATENT_DIM,
            "offs": OFFS, "beta": BETA_MNA, "lambda": LAMBDA_CONTR
        }, BEST)
    else:
        pat += 1
        if pat >= PATIENCE:
            print("Erken durdurma."); break

print("Best val:", best_val, " | saved ->", BEST)


[001] TR tot 3.9640 | c 0.9898 | n 0.9823 | con 5.3592 || VA tot 3.2188 | c 0.5318 | n 0.5609 | con 5.0376
[002] TR tot 3.6290 | c 0.9441 | n 0.9794 | con 4.7822 || VA tot 3.0565 | c 0.4945 | n 0.5426 | con 4.7985
[003] TR tot 3.4611 | c 0.8378 | n 0.9179 | con 4.6959 || VA tot 2.9672 | c 0.4601 | n 0.5171 | con 4.7038
[004] TR tot 3.2832 | c 0.7422 | n 0.8346 | con 4.5813 || VA tot 2.8934 | c 0.4514 | n 0.5055 | con 4.5808
[005] TR tot 3.1760 | c 0.7224 | n 0.7811 | con 4.4387 || VA tot 2.8034 | c 0.4504 | n 0.5030 | con 4.4043
[006] TR tot 3.1071 | c 0.7241 | n 0.7737 | con 4.3018 || VA tot 2.7565 | c 0.4448 | n 0.4970 | con 4.3252
[007] TR tot 3.0388 | c 0.7207 | n 0.7653 | con 4.1769 || VA tot 2.6939 | c 0.4339 | n 0.4897 | con 4.2263
[008] TR tot 2.9933 | c 0.6856 | n 0.7461 | con 4.1677 || VA tot 2.6555 | c 0.4207 | n 0.4791 | con 4.1822
[009] TR tot 2.9529 | c 0.6835 | n 0.7301 | con 4.1010 || VA tot 2.6225 | c 0.4076 | n 0.4639 | con 4.1515
[010] TR tot 2.8932 | c 0.6564 | n 0.

## 9. (Opsiyonel) Başarı yüzdesi ve latent çıkarımı
Aşağıdaki hücreleri istersen çalıştır. (Sunum çıktısı için faydalı.)


In [112]:
# 9) Başarı yüzdesi ve latent çıkarımı

import numpy as np, torch
from torch.serialization import add_safe_globals
add_safe_globals([np.core.multiarray._reconstruct])

# --- AE başarı yüzdesi (merkez rekonstrüksiyon MSE'ye göre) ---
def success_percent(dloader, tau=None, device=DEVICE, mdl=model):
    errs = []
    mdl.eval()
    with torch.no_grad():
        for batch in dloader:
            x_clean = batch[0].to(device)       # (B,C,L)
            y0 = mdl(x_clean)[0]                 # <-- SADECE ilk çıktıyı al (y0)
            se = (y0 - x_clean).pow(2).mean(dim=(1,2))  # pencere başı MSE
            errs.append(se.cpu().numpy())
    errs = np.concatenate(errs)
    if tau is None:
        tau = float(np.quantile(errs, 0.80))    # val için 80. persentil
    succ = float((errs < tau).mean() * 100.0)
    return succ, tau

# --- Latent çıkarımı (opsiyonel; sunum için) ---
@torch.no_grad()
def extract_latents(dloader, use_projection=True, device=DEVICE, mdl=model):
    """
    use_projection=True ise SimCLR projection head g(z) döner, değilse z (encoder çıkışı).
    Her iki durumda da L2-normalize edilmiş vektörler döndürülür.
    """
    Z = []
    mdl.eval()
    for batch in dloader:
        x_clean = batch[0].to(device)
        z = mdl.encode(x_clean)                          # (B,D)
        if use_projection:
            z = mdl.project(z)                           # g(z)
        z = torch.nn.functional.normalize(z, dim=1)
        Z.append(z.cpu().numpy())
    return np.concatenate(Z, axis=0)

# --- Kullanım örneği: Train/Val başarı yüzdesi ---
val_succ, tau = success_percent(dl_va, tau=None)
tr_succ,  _   = success_percent(dl_tr, tau=tau)
print(f"Başarı (%% MSE < {tau:.6f}) -> Train: {tr_succ:.2f}%% | Val: {val_succ:.2f}%%")

# --- (Opsiyonel) latentleri alıp dışarıda analiz etmek istersen ---
# Z_tr = extract_latents(dl_tr, use_projection=True)
# Z_va = extract_latents(dl_va, use_projection=True)
# Z_te = extract_latents(dl_te, use_projection=True)  # test loader'ın varsa


Başarı (%% MSE < 0.582400) -> Train: 68.75%% | Val: 79.69%%


## 10. Test (DIR_TE) için başarı hesabı

Amaç: Eğitim/validasyon bittikten sonra, kaydedilmiş en iyi model (best_mna.pt) ile test videolarındaki pencerelerin rekonstrüksiyon hatasına bakmak.

Pencerenin kendisini ne kadar iyi yeniden kuruyoruz?

* AE pencereyi ne kadar iyi çizebiliyor? bunun testi :

In [113]:
# ==== TEST değerlendirmesi (DIR_TE) ====
import os, numpy as np, torch
from torch.utils.data import DataLoader
from torch.serialization import add_safe_globals

# PyTorch 2.6 güvenliği
add_safe_globals([np.core.multiarray._reconstruct])

# 1) En iyi modeli ve train istatistiklerini yükle
BEST = os.path.join(OUTDIR, "best_mna.pt")
ckpt = torch.load(BEST, map_location=DEVICE, weights_only=False)

cin   = int(ckpt["cin"]); lseq  = int(ckpt["lseq"])
ldim  = int(ckpt["latent"]); offs = list(map(int, ckpt["offs"]))

model = MetricAE_MNA(cin, lseq, ldim, offs).to(DEVICE)
model.load_state_dict(ckpt["state_dict"]); model.eval()

# Train'den gelen z-score istatistikleri
stats = (ckpt["stats_mean"].numpy(), ckpt["stats_std"].numpy())

# 2) Val setini yeniden kur (τ için)
all_files = list_npy(DIR_TR)
n_val = max(1, int(len(all_files)*VAL_SPLIT))
val_files = all_files[-n_val:]

ds_val = PoseWindowDataset(val_files, WINDOW, STRIDE, fit_stats=False, stats=stats)
dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# 3) Test setini kur
test_files = list_npy(DIR_TE)
assert len(test_files) > 0, "DIR_TE içinde test .npy bulunamadı."
ds_te = PoseWindowDataset(test_files, WINDOW, STRIDE, fit_stats=False, stats=stats)
dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# 4) Başarı fonksiyonu (τ opsiyonel) —>> forward 4 değer döndürdüğü için sadece ilkini alıyoruz
def success_percent(dloader, tau=None):
    errs = []
    with torch.no_grad():
        for batch in dloader:
            x_clean = batch[0].to(DEVICE)          # (B,C,L)
            y0 = model(x_clean)[0]                  # <--- sadece ilk çıktı (merkez rekonstrüksiyon)
            se = (y0 - x_clean).pow(2).mean(dim=(1,2))  # pencere başı MSE
            errs.append(se.cpu().numpy())
    errs = np.concatenate(errs)
    if tau is None:
        tau = float(np.quantile(errs, 0.80))       # val için 80. persentil
    succ = float((errs < tau).mean() * 100.0)
    return succ, tau, errs

# 5) τ’yi val’dan al, test başarısını hesapla
val_succ, tau, _     = success_percent(dl_val, tau=None)
test_succ, _, _      = success_percent(dl_te, tau=tau)

print(f"[VAL] Başarı (% MSE < {tau:.6f}) = {val_succ:.2f}% | pencere: {len(ds_val)}")
print(f"[TEST] Başarı (% MSE < {tau:.6f}) = {test_succ:.2f}% | pencere: {len(ds_te)}")

# 6) Video-bazlı test başarısı
def per_file_success(files, tau):
    rows = []
    for fp in files:
        ds_one = PoseWindowDataset([fp], WINDOW, STRIDE, fit_stats=False, stats=stats)
        if len(ds_one) == 0:
            rows.append((os.path.basename(fp), 0.0, 0))
            continue
        dl_one = DataLoader(ds_one, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
        succ, _, _ = success_percent(dl_one, tau=tau)
        rows.append((os.path.basename(fp), succ, len(ds_one)))
    return rows

rows = per_file_success(test_files, tau)
print("\n== TEST video-bazlı başarılar ==")
for name, succ, nwin in rows:
    print(f"{name:>20s} : {succ:6.2f}%  (pencere={nwin})")


[VAL] Başarı (% MSE < 0.584085) = 79.69% | pencere: 128
[TEST] Başarı (% MSE < 0.584085) = 37.78% | pencere: 45

== TEST video-bazlı başarılar ==
       angles_61.npy :  58.33%  (pencere=12)
       angles_62.npy :  66.67%  (pencere=12)
       angles_63.npy :   0.00%  (pencere=11)
       angles_64.npy :  20.00%  (pencere=10)


## 11. Latent benzerlik başarısı (SimCLR’e uygun)
Amaç: “AE iyi çizdi mi?” yerine “encoder benzer paternleri yakın mı koymuş?” sorusunu yanıtlamak.

Encoder’ın, aynı pencerenin iki görünümünü (x1, x2) latent’te ne kadar yakınladığına baktık .

In [114]:
import torch, numpy as np, os
from torch.utils.data import DataLoader
from torch.serialization import add_safe_globals
add_safe_globals([np.core.multiarray._reconstruct])

# 1) En iyi modeli ve istatistikleri yükle
BEST = os.path.join(OUTDIR, "best_mna.pt")
ckpt = torch.load(BEST, map_location=DEVICE, weights_only=False)

cin   = int(ckpt["cin"]); lseq  = int(ckpt["lseq"])
ldim  = int(ckpt["latent"]); offs = list(map(int, ckpt["offs"]))
stats = (ckpt["stats_mean"].numpy(), ckpt["stats_std"].numpy())

model = MetricAE_MNA(cin, lseq, ldim, offs).to(DEVICE)
model.load_state_dict(ckpt["state_dict"]); model.eval()

# 2) Train/Val/Test dataset ve loader'ları (aynı z-score ile)
all_files = list_npy(DIR_TR)
n_val = max(1, int(len(all_files)*VAL_SPLIT))
val_files = all_files[-n_val:]; train_files = all_files[:-n_val]
test_files = list_npy(DIR_TE)
assert len(test_files) > 0, "DIR_TE içinde test .npy bulunamadı."

ds_tr = PoseWindowDataset(train_files, WINDOW, STRIDE, fit_stats=False, stats=stats)
ds_va = PoseWindowDataset(val_files,   WINDOW, STRIDE, fit_stats=False, stats=stats)
ds_te = PoseWindowDataset(test_files,  WINDOW, STRIDE, fit_stats=False, stats=stats)

dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
dl_va = DataLoader(ds_va, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
dl_te = DataLoader(ds_te, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

# 3) Yardımcılar: embed et, cosine hesapla
@torch.no_grad()
def embed_loader(dloader, use_aug=False):
    Z = []
    for batch in dloader:
        x_clean, x1, x2 = batch[0].to(DEVICE), batch[1].to(DEVICE), batch[2].to(DEVICE)
        X = x1 if use_aug else x_clean
        z = model.encode(X)                         # (B, D)
        z = torch.nn.functional.normalize(z, dim=1) # L2-normalize
        Z.append(z.cpu().numpy())
    return np.concatenate(Z, axis=0)

@torch.no_grad()
def val_positive_cosines(dloader):
    cos = []
    for batch in dloader:
        x1, x2 = batch[1].to(DEVICE), batch[2].to(DEVICE)
        z1 = torch.nn.functional.normalize(model.encode(x1), dim=1)
        z2 = torch.nn.functional.normalize(model.encode(x2), dim=1)
        c  = (z1*z2).sum(dim=1)                    # (B,)
        cos.append(c.cpu().numpy())
    return np.concatenate(cos, axis=0)

# 4) Eşik: validasyon pozitif çiftlerinden τ_cos
val_pos = val_positive_cosines(dl_va)
tau_cos = float(np.quantile(val_pos, 0.10))  # en düşük %10'u da "benzer" saysın
print(f"τ_cos (val pozitif 10. persentil) = {tau_cos:.4f} | val_pos mean={val_pos.mean():.4f}")

# 5) Train embed havuzu ve Test embed'leri
Z_tr = embed_loader(dl_tr, use_aug=False)   # (N_tr, D)
Z_te = embed_loader(dl_te, use_aug=False)   # (N_te, D)

# 6) Her test penceresi için train'e en yakın cosine
#    (cosine = z_test @ Z_tr^T ; max'ını al)
max_cos = []
bs = 2048
for i in range(0, len(Z_te), bs):
    z = Z_te[i:i+bs]                         # (b, D)
    sim = z @ Z_tr.T                         # (b, Ntr)
    max_cos.append(sim.max(axis=1))
max_cos = np.concatenate(max_cos, axis=0)

latent_succ = float((max_cos >= tau_cos).mean() * 100.0)
print(f"[TEST - Latent] Başarı (max cosine ≥ τ_cos) = {latent_succ:.2f}% | pencere: {len(Z_te)}")

# 7) Video-bazlı latent başarı
def per_file_latent_success(files):
    rows = []
    start = 0
    # pencere uzunlukları dosya bazlı hesaplanıyor
    lengths = []
    for fp in files:
        ds_one = PoseWindowDataset([fp], WINDOW, STRIDE, fit_stats=False, stats=stats)
        lengths.append(len(ds_one))
    idx = 0
    for fp, n in zip(files, lengths):
        if n == 0:
            rows.append((os.path.basename(fp), 0.0, 0))
            continue
        z = Z_te[idx:idx+n]                   # o videonun pencereleri
        sim = z @ Z_tr.T
        succ = float((sim.max(axis=1) >= tau_cos).mean() * 100.0)
        rows.append((os.path.basename(fp), succ, n))
        idx += n
    return rows

rows = per_file_latent_success(test_files)
print("\n== TEST video-bazlı latent başarılar ==")
for name, succ, nwin in rows:
    print(f"{name:>20s} : {succ:6.2f}%  (pencere={nwin})")


τ_cos (val pozitif 10. persentil) = 0.9932 | val_pos mean=0.9966
[TEST - Latent] Başarı (max cosine ≥ τ_cos) = 8.89% | pencere: 45

== TEST video-bazlı latent başarılar ==
       angles_61.npy :   8.33%  (pencere=12)
       angles_62.npy :  25.00%  (pencere=12)
       angles_63.npy :   0.00%  (pencere=11)
       angles_64.npy :   0.00%  (pencere=10)


## 12. Latent benzerlik: KNN tabanlı eşiğe göre test başarısı

* “Sorgu penceresi train’de bir benzer bulabiliyor mu?”

* (%84.44)? Eşik aynı protokolden kalibre edildiği için adil; SimCLR’lı encoder gerçekten benzer paternleri yakınlaştırmış.

In [115]:
# --- Latent'leri çıkar (z veya p seçilebilir) ---
@torch.no_grad()
def embed_loader(dloader, use_projection=False):  # <<< default: z
    Z = []
    model.eval()
    for batch in dloader:
        x = batch[0].to(DEVICE)
        z = model.encode(x)              # z
        if use_projection:
            z = model.project(z)         # p = g(z)
        z = torch.nn.functional.normalize(z, dim=1)
        Z.append(z.cpu().numpy())
    return np.concatenate(Z, axis=0)

# --- KNN benzerlik eşiği (val->train) ve test başarısı ---
def max_cos_to_pool(Zq, Zpool, bs=2048):
    out=[]
    for i in range(0, len(Zq), bs):
        z = Zq[i:i+bs]
        out.append(z @ Zpool.T)
    return np.max(np.concatenate(out,axis=0), axis=1)

Z_tr = embed_loader(dl_tr, use_projection=False)  # z-uzayı
Z_va = embed_loader(dl_va, use_projection=False)
Z_te = embed_loader(dl_te, use_projection=False)

val2tr = max_cos_to_pool(Z_va, Z_tr)
# kalibrasyon: val'de %90 geri çağırma için 10. persentil
tau_knn = float(np.quantile(val2tr, 0.10))
te2tr   = max_cos_to_pool(Z_te, Z_tr)
succ_knn = float((te2tr >= tau_knn).mean()*100.0)

print(f"z-uzayı: τ_knn={tau_knn:.4f} | val→tr mean={val2tr.mean():.4f}")
print(f"[TEST - Latent KNN (z)] Başarı = {succ_knn:.2f}% | pencere={len(Z_te)}")


z-uzayı: τ_knn=0.9755 | val→tr mean=0.9868
[TEST - Latent KNN (z)] Başarı = 75.56% | pencere=45


In [118]:
qs = [0.05, 0.10, 0.20, 0.25, 0.30]
print("\n== z-uzayı KNN başarı (farklı persentiller) ==")
for q in qs:
    tauq = float(np.quantile(val2tr, q))
    succ = float((te2tr >= tauq).mean()*100.0)
    print(f"q={q:>4.2f}  τ={tauq:.4f}  ->  TEST={succ:5.2f}%")



== z-uzayı KNN başarı (farklı persentiller) ==
q=0.05  τ=0.9694  ->  TEST=86.67%
q=0.10  τ=0.9755  ->  TEST=75.56%
q=0.20  τ=0.9821  ->  TEST=48.89%
q=0.25  τ=0.9838  ->  TEST=42.22%
q=0.30  τ=0.9848  ->  TEST=40.00%
